<a href="https://colab.research.google.com/github/NeoBro34/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install transformers torch bitsandbytes datasets

In [21]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# quantization (modelni o'z holicha emas hajmini kichraytirgan holda olish quantization deyiladi)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    # dtype=torch.bfloat16
)

In [19]:
# Before Fine-tunning
prompt = "Explain what a tokenizer is?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    outputs_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(outputs_ids[0], skip_special_tokens=True))

# print(model)
first_block = model.model.layers[0]
print("first_block:",first_block)
print("===========================")
print("first_block.self_attn:",first_block.self_attn)
print("===========================")
print(model.config)

Explain what a tokenizer is?
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
first_block.self_attn: LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_

In [17]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())
total_parameters = count_parameters(model)
print(f'Total parameters (including frozen 4-bit): {total_parameters:,}')

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* instruction

In [26]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
         f"{instruction}\n\n"
         "### Input:\n"
         f"{input_text}\n\n"
         "### Response:\n"
         f"{output_text}"
    )
  else:
    return(
        "### Instruction:\n"
         f"{instruction}\n\n"
         "### Response:\n"
         f"{output_text}"
    )

# generate_prompt(dataset[1])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [29]:
dataset[1]['text']

'### Instruction:\nWhat are the three primary colors?\n\n### Response:\nThe three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).'

In [32]:
dataset = dataset.select(range(7000)) # random holatda 7000ta ma'lumotni olib beradi

In [33]:
dataset = dataset.shuffle(seed=42)  # har safar ma'lumotlarni bir xil o'qishni taminlaydi

In [34]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})